In [ ]:
class EntornoPaciente:
    """Entorno realista de control de glucosa con modelo fisiológico,
    incluyendo parámetros del paciente para discretización."""
    
    def __init__(self, edad=40, peso=70, genero='H'):
        # Parámetros fisiológicos
        self.glucosa_min = 40
        self.glucosa_max = 300
        self.glucosa_objetivo = 100
        self.rango_seguro = (80, 120)
        
        # --- NUEVOS PARÁMETROS DEL PACIENTE ---
        self.edad = edad
        self.peso = peso
        self.genero = genero  # 'H' (Hombre) o 'M' (Mujer)
        
        # La sensibilidad a la insulina puede depender de estos factores
        # Se incluye una base, y se puede ajustar en reset() o paso()
        self.sensibilidad_base = 40 
        
        # Insulina
        self.insulina_activa = 0.0
        self.vida_media_insulina = 4
        self.sensibilidad_insulina = self.sensibilidad_base # Se recalcula en reset
        self.dosis_base = 1.0
        
        # Estado actual
        self.glucosa_actual = 150
        self.tiempo_hora = 0
        self.comida_reciente = False
        
        # Acciones
        self.acciones = {
            'AUM_DOSIS': 1.5,
            'MANTENER': 1.0,
            'RED_DOSIS': 0.5,
            'NO_INSULINA': 0.0
        }
        
        # Historial
        self.historial_glucosa = deque(maxlen=100)
        self.historial_acciones = deque(maxlen=100)
        self.historial_recompensas = deque(maxlen=100)
        
        self.ruido_metabolico = 5
    
    def reset(self):
        """Reinicia con estado inicial aleatorio y recalcula la sensibilidad."""
        
        # --- SIMULACIÓN DE PACIENTES DIVERSOS EN CADA EPISODIO ---
        # Puedes fijar estos valores si quieres un paciente estático
        # o dejar la aleatoriedad para entrenar a un agente más robusto.
        self.edad = np.random.randint(20, 80)
        self.peso = np.random.uniform(50, 100)
        self.genero = np.random.choice(['H', 'M'])
        
        # Ajuste simple de sensibilidad (ejemplo)
        # Mujeres (M) suelen tener una sensibilidad ligeramente mayor (menos resistencia)
        ajuste_genero = 1.0 if self.genero == 'H' else 1.1 
        # Jóvenes o delgados pueden ser más sensibles
        ajuste_edad_peso = 1 + ((80 - self.edad) / 100) * ((100 - self.peso) / 100) * 0.5
        self.sensibilidad_insulina = self.sensibilidad_base * ajuste_genero * ajuste_edad_peso
        # Asegurarse de que no sea un valor absurdo
        self.sensibilidad_insulina = np.clip(self.sensibilidad_insulina, 20, 70) 

        self.glucosa_actual = np.random.uniform(140, 200)
        self.tiempo_hora = np.random.randint(0, 24)
        self.insulina_activa = 0.0
        self.comida_reciente = np.random.choice([True, False])
        
        self.historial_glucosa.clear()
        self.historial_acciones.clear()
        self.historial_recompensas.clear()
        
        return self._obtener_estado()
    
    def _obtener_estado(self):
        """Estado discretizado, incluyendo edad, peso y género."""
        
        # 1. Nivel de Glucosa (0: BAJA, 1: NORMAL, 2: ALTA)
        if self.glucosa_actual < 70:
            glucosa_nivel = 0
        elif self.glucosa_actual <= 140:
            glucosa_nivel = 1
        else:
            glucosa_nivel = 2
        
        # 2. Periodo del Día (0-3: cuartos del día)
        periodo = self.tiempo_hora // 6
        
        # 3. Comida Reciente (0: No, 1: Sí)
        comida = 1 if self.comida_reciente else 0
        
        # 4. Edad Discretizada (0: Joven, 1: Adulto, 2: Mayor)
        if self.edad < 40:
            edad_nivel = 0
        elif self.edad <= 65:
            edad_nivel = 1
        else:
            edad_nivel = 2
            
        # 5. Peso Discretizado (0: Delgado, 1: Normal, 2: Sobrepeso/Obeso)
        # Uso de IMC simplificado: Peso / (1.75**2) ~ 22.86,
        # Asumiendo una altura media de 1.75m.
        # <60kg -> Delgado (IMC < 20)
        # 60-85kg -> Normal (IMC 20-28)
        # >85kg -> Obeso (IMC > 28)
        if self.peso < 60:
            peso_nivel = 0
        elif self.peso <= 85:
            peso_nivel = 1
        else:
            peso_nivel = 2
            
        # 6. Género (0: Hombre 'H', 1: Mujer 'M')
        genero_codificado = 1 if self.genero == 'M' else 0
        
        # El estado es una tupla de 6 elementos
        return (glucosa_nivel, periodo, comida, edad_nivel, peso_nivel, genero_codificado)
    
    # El resto de los métodos se mantienen igual, solo se actualiza la sensibilidad
    # y la glucosa actual en base a los nuevos parámetros en reset y paso.
    
    def obtener_clave_estado(self):
        """Estado como string para la tabla Q."""
        estado = self._obtener_estado()
        return ",".join(map(str, estado))

    def obtener_acciones_validas(self):
        """Lista de acciones."""
        return list(self.acciones.keys())
    
    def obtener_info_estado(self):
        """Info legible del estado."""
        if self.glucosa_actual < 70:
            nivel = "BAJA"
        elif self.glucosa_actual <= 140:
            nivel = "NORMAL"
        else:
            nivel = "ALTA"
        
        return nivel, f"{self.tiempo_hora:02d}:00", f"{self.glucosa_actual:.1f} mg/dL"

    def paso(self, accion_nombre):
        """Simula un paso temporal."""
        # Aplicar insulina
        dosis = self.acciones[accion_nombre] * self.dosis_base
        self.insulina_activa += dosis
        
        # Efecto insulina (depende de la sensibilidad ajustada en reset)
        reduccion = self.insulina_activa * self.sensibilidad_insulina / 100.0 # Normalización
        self.glucosa_actual -= reduccion
        
        # Degradación insulina
        self.insulina_activa *= (0.5 ** (1/self.vida_media_insulina))
        
        # Producción hepática (puede ser ajustada por peso/edad/género si es necesario)
        self.glucosa_actual += 5
        
        # Comida
        if self.comida_reciente:
            self.glucosa_actual += 30
            self.comida_reciente = False
        
        if np.random.random() < 0.1:
            self.comida_reciente = True
        
        # Ruido
        self.glucosa_actual += np.random.uniform(-self.ruido_metabolico, 
                                                  self.ruido_metabolico)
        
        # Límites
        self.glucosa_actual = np.clip(self.glucosa_actual, 
                                       self.glucosa_min, 
                                       self.glucosa_max)
        
        self.tiempo_hora = (self.tiempo_hora + 1) % 24
        
        # Recompensa
        recompensa = self._calcular_recompensa(accion_nombre)
        
        # Historial
        self.historial_glucosa.append(self.glucosa_actual)
        self.historial_acciones.append(accion_nombre)
        self.historial_recompensas.append(recompensa)
        
        terminado = self.glucosa_actual < 50
        
        return self.obtener_clave_estado(), recompensa, terminado
    
    def _calcular_recompensa(self, accion):
        """Sistema de recompensas realista."""
        r = 0
        
        # En rango óptimo
        if self.rango_seguro[0] <= self.glucosa_actual <= self.rango_seguro[1]:
            r += 10
            dist = abs(self.glucosa_actual - self.glucosa_objetivo)
            r += max(0, 5 - dist / 4)
        
        # Hipoglucemia
        if self.glucosa_actual < 70:
            r -= (70 - self.glucosa_actual) * 0.5
            if self.glucosa_actual < 50:
                r -= 50
        
        # Hiperglucemia
        if self.glucosa_actual > 180:
            r -= (self.glucosa_actual - 180) * 0.3
        
        # Variabilidad
        if len(self.historial_glucosa) >= 2:
            cambio = abs(self.glucosa_actual - self.historial_glucosa[-1])
            if cambio > 30:
                r -= 2
        
        # Eficiencia
        if self.rango_seguro[0] <= self.glucosa_actual <= self.rango_seguro[1]:
            if accion in ['MANTENER', 'NO_INSULINA']:
                r += 2
        
        r -= 0.1  # Penalización por paso
        
        return r

Explicación de la DiscretizaciónLa función _obtener_estado ahora genera una tupla de 6 elementos, lo que aumenta el espacio de estados de la tabla Q.El estado del paciente es ahora:$$\text{Estado} = (\text{GlucosaNivel}, \text{PeriodoDía}, \text{Comida}, \text{EdadNivel}, \text{PesoNivel}, \text{Género})$$VariableNiveles (Discretización)Rango Aprox.GlucosaNivel30 (< 70), 1 (70-140), 2 (> 140)PeriodoDía40 (00-05), 1 (06-11), 2 (12-17), 3 (18-23)Comida20 (No), 1 (Sí)EdadNivel30 (< 40), 1 (40-65), 2 (> 65)PesoNivel30 (< 60kg), 1 (60-85kg), 2 (> 85kg)Género20 (Hombre 'H'), 1 (Mujer 'M')El espacio total de estados es ahora:$$3 \times 4 \times 2 \times 3 \times 3 \times 2 = \mathbf{432} \text{ estados}$$Este aumento en el espacio de estados es manejable y permite al agente aprender políticas específicas para pacientes con diferentes características demográficas. Por ejemplo, el agente aprenderá que para un paciente joven y delgado (mayor sensibilidad), la dosis de insulina debe ser más cautelosa que para un paciente mayor y con sobrepeso (menor sensibilidad/mayor resistencia).¿Te gustaría que ejecutemos un nuevo entrenamiento con este entorno ampliado, o prefieres enfocarte en otro aspecto del código?


¡Claro! Para incorporar la edad, el peso y el género en la discretización del entorno para el agente Q-Learning, necesitamos modificar la clase EntornoPaciente.

La estrategia será:

Modificar __init__: Agregar nuevos parámetros para el paciente (edad, peso, género) y la discretización.

Modificar reset: Inicializar el estado del paciente (glucosa, tiempo, comida) y los nuevos parámetros de forma fija o aleatoria (para simular diferentes pacientes).

Modificar _obtener_estado: Discretizar los nuevos parámetros para incluirlos en el estado que el agente percibe.

💻 Código Modificado
A continuación, se muestra la clase EntornoPaciente con las modificaciones. Los cambios se enfocan en la sección 2 del notebook que subiste.

Python


Por Qué se Aumentaron las Variables (Edad, Peso, Género)Tu sistema original modelaba a un "paciente genérico". Sin embargo, la diabetes y el metabolismo de la insulina varían enormemente.Tu ingeniero te pidió incluir Edad, Peso y Género porque estos son factores biológicos cruciales que afectan directamente la sensibilidad a la insulina y la velocidad con que el cuerpo procesa la glucosa:Edad: El metabolismo basal y la sensibilidad a la insulina suelen disminuir con la edad, requiriendo diferentes dosis.Peso: Un mayor peso corporal o una mayor masa grasa (obesidad) se asocia típicamente con la resistencia a la insulina, lo que significa que se necesita una dosis más alta para lograr el mismo efecto.Género: Existen diferencias hormonales que pueden influir en la sensibilidad a la insulina (por ejemplo, las mujeres a veces tienen una sensibilidad basal ligeramente diferente o fluctuante debido a ciclos hormonales).Al agregar estas variables al estado del agente, el modelo ya no trata a todos los pacientes como iguales. El agente puede aprender una política: "Si la glucosa es ALTA y es de noche, aplica Dosis A a un paciente JOVEN/DELGADO, pero aplica Dosis B (más alta) a un paciente MAYOR/OBESO."🧠 ¿En Qué Ayuda la Discretización?La discretización es el paso clave que permite que el Aprendizaje por Refuerzo (RL) funcione con variables continuas o complejas, y es crucial para tu algoritmo Q-Learning:1. Requerimiento del Q-LearningEl algoritmo Q-Learning utiliza una Tabla Q (Q-table) para almacenar el "valor" de cada acción en cada estado. Esta tabla, por definición, requiere estados discretos (finitos) que puedan usarse como claves o índices.Estado (Clave)Acción (AUM_DOSIS)Acción (RED_DOSIS)...ALTA, NOCHE, COMIDA_SI, MAYOR, OBESO, HOMBRE15.210.1...NORMAL, DÍA, COMIDA_NO, JOVEN, NORMAL, MUJER20.518.9...Si usáramos la edad real (por ejemplo, 35.7 años, 35.8 años, etc.), el número de estados sería infinito, y la tabla Q nunca podría completarse.2. Generalización (Agrupación Inteligente)La discretización agrupa rangos biológicamente similares:En lugar de tratar individualmente a un paciente de 40 años, 41 años y 42 años, los agrupamos en el nivel ADULTO (edad_nivel = 1).Esto permite al agente generalizar su experiencia de entrenamiento, aplicando la misma política aprendida a todo el rango "ADULTO".3. Aumento del Espacio de EstadosGracias a la discretización, has ampliado tu espacio de estados de 24 estados (solo glucosa, tiempo, comida) a 432 estados (incluyendo edad, peso, género).$$3 \text{(Glucosa)} \times 4 \text{(Tiempo)} \times 2 \text{(Comida)} \times 3 \text{(Edad)} \times 3 \text{(Peso)} \times 2 \text{(Género)} = \mathbf{432}$$Este aumento es bueno porque le da al agente el contexto necesario para tomar decisiones más matizadas, resultando en una política de tratamiento que se adapta a diferentes perfiles de pacientes.